# Syncing a table to the Hive Metastore (HMS) with Apache XTable™
This notebook walks through the flow described in the
[Creating your first interoperable table](https://xtable.apache.org/docs/how-to) and
[Hive Metastore](https://xtable.apache.org/docs/hms) guides:

1. Create a Hudi source table with Spark.
2. Use XTable to sync the table metadata to Delta Lake and Iceberg (no data is copied or rewritten).
3. Register the synced tables in the Hive Metastore.
4. Query the same data from Trino and Presto in all three formats.

In [ ]:
import org.apache.spark.sql._
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
import java.util._
import java.sql._
import org.apache.xtable.conversion._
import org.apache.xtable.model.sync._
import org.apache.xtable.model.storage._
import org.apache.xtable.hudi._

## Step 1: Create a Hudi source table
We create a small `people` dataset partitioned by `city`, mirroring the
[how-to guide](https://xtable.apache.org/docs/how-to#create-dataset).

In [ ]:
val peopleTableName = "people"
val peopleBasePath = "file:/home/data/db/people"

val spark = org.apache.spark.sql.SparkSession.builder()
    .appName("xtable-hms-demo")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .master("local")
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

val peopleDf = Seq(
    (1, "John", 25, "NYC", "2023-09-28 00:00:00"),
    (2, "Emily", 30, "SFO", "2023-09-28 00:00:00"),
    (3, "Michael", 35, "ORD", "2023-09-28 00:00:00"),
    (4, "Andrew", 40, "NYC", "2023-10-28 00:00:00"),
    (5, "Bob", 28, "SEA", "2023-09-23 00:00:00"),
    (6, "Charlie", 31, "DFW", "2023-08-29 00:00:00")
  ).toDF("id", "name", "age", "city", "create_ts")

val peopleWriteOptions = new HashMap[String, String]()
peopleWriteOptions.put("hoodie.table.name", peopleTableName)
peopleWriteOptions.put("hoodie.datasource.write.recordkey.field", "id")
peopleWriteOptions.put("hoodie.datasource.write.partitionpath.field", "city")
peopleWriteOptions.put("hoodie.datasource.write.precombine.field", "create_ts")
peopleWriteOptions.put("hoodie.datasource.write.hive_style_partitioning", "true")
// Write in table version 6 so older Hudi readers (e.g. the Presto Hudi connector) can also read the table.
peopleWriteOptions.put("hoodie.write.table.version", "6")

peopleDf.write.format("hudi").options(peopleWriteOptions).mode("overwrite").save(peopleBasePath)

spark.read.format("hudi").load(peopleBasePath).select("id", "name", "age", "city").show()

## Step 2: Sync the table to Delta Lake and Iceberg with XTable
XTable translates the Hudi table metadata into Delta Lake and Iceberg metadata in place.
For a partitioned Hudi source, the partition layout is described with
`xtable.hudi.source.partition_field_spec_config` (here: identity partitioning on `city`).

In [ ]:
val sourceProperties = new Properties()
sourceProperties.setProperty("xtable.hudi.source.partition_field_spec_config", "city:VALUE")

val peopleSourceTable = SourceTable.builder()
    .name(peopleTableName)
    .formatName(TableFormat.HUDI)
    .basePath(peopleBasePath)
    .additionalProperties(sourceProperties)
    .build()

val peopleDeltaTarget = TargetTable.builder()
    .name(peopleTableName)
    .formatName(TableFormat.DELTA)
    .basePath(peopleBasePath)
    .additionalProperties(new Properties())
    .build()

val peopleIcebergTarget = TargetTable.builder()
    .name(peopleTableName)
    .formatName(TableFormat.ICEBERG)
    .basePath(peopleBasePath)
    .additionalProperties(new Properties())
    .build()

val peopleConversionConfig = ConversionConfig.builder()
    .sourceTable(peopleSourceTable)
    .targetTables(Arrays.asList(peopleDeltaTarget, peopleIcebergTarget))
    .syncMode(SyncMode.INCREMENTAL)
    .build()

val hudiConversionSourceProvider = new HudiConversionSourceProvider()
hudiConversionSourceProvider.init(spark.sparkContext.hadoopConfiguration)

val conversionController = new ConversionController(spark.sparkContext.hadoopConfiguration)
conversionController.sync(peopleConversionConfig, hudiConversionSourceProvider)

## Step 3: Register the synced tables in the Hive Metastore
The synced Delta and Iceberg metadata now lives next to the data. To make the tables visible to query
engines we register them in HMS using Trino's `register_table` procedures (the same registration can be
done with Spark DDL as shown in the [HMS guide](https://xtable.apache.org/docs/hms)).

In [ ]:
val trinoConnection = DriverManager.getConnection("jdbc:trino://trino:8080?user=admin")
val trinoStatement = trinoConnection.createStatement()

def runTrino(sql: String): Unit = {
  trinoStatement.execute(sql)
  println(s"OK: $sql")
}

// Metadata-only operation: removes a stale registration from a previous run
// without touching any data files.
def unregisterQuietly(sql: String): Unit = {
  try {
    trinoStatement.execute(sql)
    println(s"Unregistered stale entry: $sql")
  } catch {
    case e: SQLException => // nothing to unregister
  }
}

runTrino("CREATE SCHEMA IF NOT EXISTS iceberg.hms_demo")

unregisterQuietly("CALL iceberg.system.unregister_table(schema_name => 'hms_demo', table_name => 'people_iceberg')")
unregisterQuietly("CALL delta.system.unregister_table(schema_name => 'hms_demo', table_name => 'people_delta')")

runTrino("CALL iceberg.system.register_table(schema_name => 'hms_demo', table_name => 'people_iceberg', table_location => 'file:///home/data/db/people')")
runTrino("CALL delta.system.register_table(schema_name => 'hms_demo', table_name => 'people_delta', table_location => 'file:///home/data/db/people')")

## Step 4: Query the synced tables from Trino
The same data files are now queryable as Iceberg and Delta tables.

In [ ]:
def runTrinoQuery(query: String): Unit = {
  println(s"> $query")
  val rs = trinoStatement.executeQuery(query)
  val columnCount = rs.getMetaData.getColumnCount
  println((1 to columnCount).map(rs.getMetaData.getColumnName).mkString(" | "))
  while (rs.next()) {
    println((1 to columnCount).map(rs.getString).mkString(" | "))
  }
  rs.close()
  println()
}

runTrinoQuery("SELECT COUNT(*) AS row_count FROM iceberg.hms_demo.people_iceberg")
runTrinoQuery("SELECT COUNT(*) AS row_count FROM delta.hms_demo.people_delta")
runTrinoQuery("SELECT city, COUNT(*) AS people FROM iceberg.hms_demo.people_iceberg GROUP BY city ORDER BY city")
runTrinoQuery("SELECT id, name, age, city FROM delta.hms_demo.people_delta ORDER BY id")

## Step 5: Register the Hudi table in HMS and query from Presto
The Hudi source table itself is registered in HMS with Hudi's Hive sync
(see [Hudi Hive Metastore docs](https://hudi.apache.org/docs/syncing_metastore)), after which Presto can
query it through its `hudi` connector. Presto can also query the Iceberg table registered above.

In [ ]:
import org.apache.hudi.common.config.TypedProperties
import org.apache.hudi.hive.{HiveSyncTool, HiveSyncConfigHolder}
import org.apache.hudi.sync.common.HoodieSyncConfig

val hiveSyncProps = new TypedProperties()
hiveSyncProps.setProperty(HiveSyncConfigHolder.HIVE_SYNC_MODE.key, "hms")
hiveSyncProps.setProperty(HiveSyncConfigHolder.METASTORE_URIS.key, "thrift://hive-metastore:9083")
hiveSyncProps.setProperty(HoodieSyncConfig.META_SYNC_DATABASE_NAME.key, "hms_demo")
hiveSyncProps.setProperty(HoodieSyncConfig.META_SYNC_TABLE_NAME.key, "people_hudi")
hiveSyncProps.setProperty(HoodieSyncConfig.META_SYNC_BASE_PATH.key, peopleBasePath)
hiveSyncProps.setProperty(HoodieSyncConfig.META_SYNC_PARTITION_FIELDS.key, "city")
hiveSyncProps.setProperty(HoodieSyncConfig.META_SYNC_PARTITION_EXTRACTOR_CLASS.key, "org.apache.hudi.hive.HiveStylePartitionValueExtractor")

val hiveSyncTool = new HiveSyncTool(hiveSyncProps, spark.sparkContext.hadoopConfiguration)
hiveSyncTool.syncHoodieTable()
println("Hudi table synced to HMS as hms_demo.people_hudi")

In [ ]:
val prestoConnection = DriverManager.getConnection("jdbc:presto://presto:8082/hudi/hms_demo?user=admin")
val prestoStatement = prestoConnection.createStatement()

def runPrestoQuery(query: String): Unit = {
  println(s"> $query")
  val rs = prestoStatement.executeQuery(query)
  val columnCount = rs.getMetaData.getColumnCount
  println((1 to columnCount).map(rs.getMetaData.getColumnName).mkString(" | "))
  while (rs.next()) {
    println((1 to columnCount).map(rs.getString).mkString(" | "))
  }
  rs.close()
  println()
}

runPrestoQuery("SELECT COUNT(*) AS row_count FROM hudi.hms_demo.people_hudi")
runPrestoQuery("SELECT id, name, age, city FROM hudi.hms_demo.people_hudi ORDER BY id")
runPrestoQuery("SELECT city, COUNT(*) AS people FROM iceberg.hms_demo.people_iceberg GROUP BY city ORDER BY city")

prestoStatement.close()
prestoConnection.close()

## Recap
One copy of the data under `/home/data/db/people` is now registered in the Hive Metastore three ways —
`hms_demo.people_hudi` (Hudi), `hms_demo.people_delta` (Delta Lake) and `hms_demo.people_iceberg`
(Iceberg) — and is queryable from Spark, Trino and Presto without copying or rewriting any data files.